In [1]:
import pandas as pd
import os
from scipy.spatial.transform import Rotation as R

In [2]:
def global2local(gyro_global, acc_global, quats):
    """
    Rotate gyroscope and acceleration data from global frame to sensor's local frame.

    Parameters:
        gyro_global (Nx3 array): angular velocity in global frame
        acc_global (Nx3 array): linear acceleration in global frame
        quats (Nx4 array): quaternions representing sensor orientation (w, x, y, z)

    Returns:
        gyro_local (Nx3 array): angular velocity in local frame
        acc_local (Nx3 array): linear acceleration in local frame
    """
    # Convert quaternion from (w, x, y, z) to (x, y, z, w) for scipy
    quats_xyzw = quats[:, [1, 2, 3, 0]]

    # Create Rotation object
    r_global_to_local = R.from_quat(quats_xyzw).inv()

    # Rotate gyroscope and acceleration from global to local
    gyro_local = r_global_to_local.apply(gyro_global)
    acc_local = r_global_to_local.apply(acc_global)

    return gyro_local, acc_local

In [ ]:
def correct_all_segments_orientation(folder_path):
    """
    Applies orientation correction to all sensor segments in the dataset.

    Parameters:
        folder_path (String): Data file path.
    """

    print("Working on", folder_path)   
    df = pd.read_csv(os.path.join(folder_path, "merged.csv"))

    segments = [
        'Pelvis', 'L5', 'L3', 'T12', 'T8', 'Neck', 'Head',
        'RightShoulder', 'RightUpperArm', 'RightForeArm', 'RightHand',
        'LeftShoulder', 'LeftUpperArm', 'LeftForeArm', 'LeftHand',
        'RightUpperLeg', 'RightLowerLeg', 'RightFoot', 'RightToe',
        'LeftUpperLeg', 'LeftLowerLeg', 'LeftFoot', 'LeftToe'
    ]

    all_local_data = []

    for segment in segments:
        acc_cols = [f'acceleration_{segment}_x', f'acceleration_{segment}_y', f'acceleration_{segment}_z']
        gyro_cols = [f'angularVelocity_{segment}_x', f'angularVelocity_{segment}_y', f'angularVelocity_{segment}_z']
        quat_cols = [f'orientation_{segment}_q1', f'orientation_{segment}_qi', f'orientation_{segment}_qj', f'orientation_{segment}_qk']

        # Check if all required columns exist
        if all(col in df.columns for col in acc_cols + gyro_cols + quat_cols):
            acc_global = df[acc_cols].values
            gyro_global = df[gyro_cols].values
            quats = df[quat_cols].values

            gyro_local, acc_local = global2local(gyro_global, acc_global, quats)

            # Create DataFrames with appropriate column names
            gyro_local_df = pd.DataFrame(gyro_local, columns=[f'{col}_local' for col in gyro_cols], index=df.index)
            acc_local_df = pd.DataFrame(acc_local, columns=[f'{col}_local' for col in acc_cols], index=df.index)

            all_local_data.append(gyro_local_df)
            all_local_data.append(acc_local_df)
        else:
            print(f"Skipping segment {segment}: missing columns.")

    df_local = pd.concat([df] + all_local_data, axis=1)

    df_local.to_csv(os.path.join(folder_path, "merged.csv"), index=False)

In [ ]:
dataset_path = "data_set"

print("Adding local sensor orientation...")
for course_folder in os.listdir(dataset_path):
    course_folder_path = os.path.join(dataset_path, course_folder)
    if os.path.isdir(course_folder_path):
        for subfolder in os.listdir(course_folder_path):
            subfolder_path = os.path.join(course_folder_path, subfolder)
            if os.path.isdir(subfolder_path):
                correct_all_segments_orientation(subfolder_path)
print("Local sensor orientation added successfully")